In [1]:
!nvidia-smi

Wed May 20 11:07:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.44                 Driver Version: 591.44         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   42C    P3              9W /   35W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!pip install uv

In [3]:
!uv pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

Using Python 3.10.19 environment at: D:\Anaconda3\envs\cudnn-env
Resolved 6 packages in 1m 55s
 Downloaded llama-cpp-python
Prepared 1 package in 1m 30s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 7.94s
 + diskcache==5.6.3
 + llama-cpp-python==0.3.23


In [4]:
!uv pip show llama-cpp-python

Name: llama-cpp-python
Version: 0.3.23
Location: D:\Anaconda3\envs\cudnn-env\Lib\site-packages
Requires: diskcache, jinja2, numpy, typing-extensions
Required-by:


Using Python 3.10.19 environment at: D:\Anaconda3\envs\cudnn-env


In [6]:
!uv pip install hf_xet

Using Python 3.10.19 environment at: D:\Anaconda3\envs\cudnn-env
Resolved 1 package in 187ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 73ms
 + hf-xet==1.5.0


In [1]:
!uv pip install ipywidgets

Using Python 3.10.19 environment at: D:\Anaconda3\envs\cudnn-env
Resolved 20 packages in 623ms
 Downloaded widgetsnbextension
Prepared 3 packages in 680ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 3 packages in 115ms
 + ipywidgets==8.1.8
 + jupyterlab-widgets==3.0.16
 + widgetsnbextension==4.0.15


In [2]:
# !pip install llama-cpp-python

from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="Viraj0112/gemma4-finetuned-q4",
	filename="gemma4_Q4_K_M.gguf",
)

./gemma4_Q4_K_M.gguf:   0%|          | 0.00/3.43G [00:00<?, ?B/s]

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 6143 MiB):
  Device 0: NVIDIA GeForce RTX 3050 6GB Laptop GPU, compute capability 8.6, VMM: yes, VRAM: 6143 MiB
llama_model_loader: loaded meta data with 39 key-value pairs and 601 tensors from C:\Users\Viraaj Sawant\.cache\huggingface\hub\models--Viraj0112--gemma4-finetuned-q4\snapshots\3c58d72a99a5b87f123b4326e50b95cbda49e989\.\gemma4_Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma4
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 4.6B
llama_model_loader: - kv   4:                         gemma4.block_count u3

In [5]:
!uv pip install transformers datasets accelerate bitsandbytes peft trl

Using Python 3.10.19 environment at: D:\Anaconda3\envs\cudnn-env
Resolved 54 packages in 1.10s
 Downloaded bitsandbytes
Prepared 2 packages in 8.61s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 493ms
 + bitsandbytes==0.49.2
 + peft==0.19.1


In [ ]:
response = llm.create_chat_completion(
    messages=[
        {
            "role": "system", 
            "content": "You are a specialized mental health assistant. You must ONLY answer questions related to mental health, psychology, and emotional well-being. If a user asks a question about general knowledge, geography, math, coding, or any other unrelated topic, you must politely decline to answer and remind them of your purpose."
        },
        {
            "role": "user", 
            "content": "WHAT IS THE CAPITAL OF FRANCE?"
        }
    ]
)

print(response['choices'][0]['message']['content'])


In [11]:
import re

# Assuming 'llm' is already initialized via Llama()

GEN_CONFIG = {
    "max_tokens": 512,
    "temperature": 0.65,
    "top_p": 0.9,
    "repeat_penalty": 1.05, # Equates to repetition_penalty
}

# Initialize messages with the system prompt to enforce domain limits
system_prompt = {
    "role": "system", 
    "content": "You are a specialized mental health assistant. You must ONLY answer questions related to mental health, psychology, and emotional well-being. If a user asks a question about general knowledge, geography, math, coding, or any other unrelated topic, you must politely decline to answer and remind them of your purpose."
}

messages = [system_prompt]
print("Type 'exit' to stop.\n")

while True:
    user_input = input("User: ").strip()
    if user_input.lower() == "exit":
        print("\nChat ended.")
        break

    # Add user message to history
    messages.append({"role": "user", "content": user_input})

    # Generate response using llama_cpp
    response_data = llm.create_chat_completion(
        messages=messages,
        **GEN_CONFIG
    )

    # Extract the text content from the response
    response = response_data['choices'][0]['message']['content'].strip()

    # Apply the same text cleaning formatting you had
    response = re.sub(r'\*+', '', response)
    response = re.sub(r'#{1,3} ', '', response)
    response = re.sub(r'[-—]{3,}', '', response)
    response = re.sub(r'[^\x00-\x7F]', '', response)

    if not response:
        response = "I'm listening."

    print(f"\nAssistant: {response}\n")

    # Add assistant response to history so the model remembers the conversation
    messages.append({"role": "assistant", "content": response})

    # Trim history by message count to save memory
    # We keep index 0 (System Prompt) and grab the last 19 messages
    if len(messages) > 20:
        messages = [messages[0]] + messages[-19:]


Type 'exit' to stop.



Llama.generate: 70 prefix-match hit, remaining 10 prompt tokens to eval
CUDA Graph id 2163 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 2164 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CUDA Graph id 2164 reused
CUDA Graph id 2163 reused
CU


Assistant: That sounds wonderful! It's lovely when you experience such genuine happiness. Would you like to share what is making you feel this way?



Llama.generate: 108 prefix-match hit, remaining 19 prompt tokens to eval
CUDA Graph id 2251 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 2252 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
CUDA Graph id 2252 reused
CUDA Graph id 2251 reused
C


Assistant: That is fantastic news! Completing a project can bring such a profound sense of accomplishment and joy. What kind of project was it?



...


Assistant: That feeling of seeing your hard work translate into something functional and successful is truly rewarding. It speaks volumes about your skills and dedication. Congratulations on such a successful completion!



Llama.generate: 288 prefix-match hit, remaining 40 prompt tokens to eval
CUDA Graph id 3005 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 3006 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
CUDA Graph id 3006 reused
CUDA Graph id 3005 reused
C


Assistant: I am so sorry that you are feeling this deep sadness right now. Please know that your feelings are valid, even when things have gone well.

I want to assure you that I am here to listen without judgment and offer support regarding what you are feeling. If you would like to talk more about why you feel this way, I am here for you.



Llama.generate: 400 prefix-match hit, remaining 15 prompt tokens to eval
CUDA Graph id 3093 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 3094 reused
ggml_backend_cuda_graph_compute: CUDA graph warmup complete
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
CUDA Graph id 3094 reused
CUDA Graph id 3093 reused
C


Assistant: Our last conversation was about you feeling very happy because you completed a Kaggle hackathon and your model is working well, and then you expressed feeling very sad afterward.

Is there anything specific from that time you would like to revisit or discuss further?


Chat ended.
